In [ ]:
# Enable inline plotting
%matplotlib inline

In [ ]:
# Install required libraries if not already present
!pip install anndata scanpy==1.11.1 squidpy leidenalg hdbscan==0.8.29 pandas==2.2.2

# Import required libraries
import numpy as np
import pandas as pd

import anndata as ad
import scanpy as sc
import squidpy as sq

In [ ]:
# Print package versions
sc.logging.print_header()
print(f"squidpy=={sq.__version__}")

In [ ]:
# Load Visium H&E image dataset
img = sq.datasets.visium_hne_image()

# Load preprocessed AnnData object
adata = sq.datasets.visium_hne_adata()

In [ ]:
# Visualize spatial clusters
sq.pl.spatial_scatter(
    adata,
    color="cluster"
)

In [ ]:
# Calculate image features at multiple scales
for scale in [1.0, 2.0]:

    feature_name = f"features_summary_scale{scale}"

    sq.im.calculate_image_features(
        adata,
        img.compute(),
        features="summary",
        key_added=feature_name,
        n_jobs=4,
        scale=scale,
    )

In [ ]:
# Combine extracted image features into one dataframe
adata.obsm["features"] = pd.concat(
    [
        adata.obsm[f]
        for f in adata.obsm.keys()
        if "features_summary" in f
    ],
    axis="columns",
)

# Remove duplicate feature names
adata.obsm["features"].columns = ad.utils.make_index_unique(
    adata.obsm["features"].columns
)

In [ ]:
# Helper function for clustering image features
def cluster_features(features: pd.DataFrame, like=None) -> pd.Series:
    """
    Calculate Leiden clustering of features.
    """

    # Filter features if keyword is provided
    if like is not None:
        features = features.filter(like=like)

    # Create temporary AnnData object
    adata_tmp = ad.AnnData(features)

    # Scale features
    sc.pp.scale(adata_tmp)

    # PCA
    sc.pp.pca(
        adata_tmp,
        n_comps=min(10, features.shape[1] - 1)
    )

    # Compute neighbors
    sc.pp.neighbors(adata_tmp)

    # Leiden clustering
    sc.tl.leiden(adata_tmp)

    return adata_tmp.obs["leiden"]

In [ ]:
# Generate feature-based clusters
adata.obs["features_cluster"] = cluster_features(
    adata.obsm["features"],
    like="summary"
)

In [ ]:
# Compare image-feature clusters with gene-expression clusters
sq.pl.spatial_scatter(
    adata,
    color=["features_cluster", "cluster"]
)

In [ ]:
# Build spatial neighbor graph
sq.gr.spatial_neighbors(adata)

In [ ]:
# Compute neighborhood enrichment
sq.gr.nhood_enrichment(
    adata,
    cluster_key="cluster"
)

In [ ]:
# Compute co-occurrence statistics
sq.gr.co_occurrence(
    adata,
    cluster_key="cluster"
)

In [ ]:
# Visualize co-occurrence analysis
sq.pl.co_occurrence(
    adata,
    cluster_key="cluster",
    clusters="Hippocampus",
    figsize=(8, 4),
)

In [ ]:
# Ligand-receptor interaction analysis
sq.gr.ligrec(
    adata,
    n_perms=100,
    cluster_key="cluster",
)

In [ ]:
# Visualize ligand-receptor interactions
sq.pl.ligrec(
    adata,
    cluster_key="cluster",
    source_groups="Hippocampus",
    target_groups=[
        "Pyramidal_layer",
        "Pyramidal_layer_dentate_gyrus",
    ],
    means_range=(3, np.inf),
    alpha=1e-4,
    swap_axes=True,
)

In [ ]:
# Select highly variable genes
genes = adata[:, adata.var.highly_variable].var_names.values[:1000]

In [ ]:
# Compute Moran's I spatial autocorrelation
sq.gr.spatial_autocorr(
    adata,
    mode="moran",
    genes=genes,
    n_perms=100,
    n_jobs=1,
)

In [ ]:
# Display top spatially variable genes
adata.uns["moranI"].head(10)

In [ ]:
# Visualize spatial expression of selected genes
sq.pl.spatial_scatter(
    adata,
    color=["Olfm1", "Plp1", "Itpka", "cluster"]
)